In [ ]:
import os

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

from IPython.display import clear_output


In [ ]:
# Caminhos dos arquivos principais
caminho_triagem = "../data/processed/triagem.csv"
caminho_dataset = "../data/processed/dataset_selecionado.csv"

# Carrega os dados
triagem = pd.read_csv(caminho_triagem)
dataset = pd.read_csv(caminho_dataset)

# Padroniza os IDs como texto sem ".0"
triagem["id"] = (
    triagem["id"]
    .astype(str)
    .str.replace(".0", "", regex=False)
)

dataset["id"] = (
    dataset["id"]
    .astype(str)
    .str.replace(".0", "", regex=False)
)

print(f"Imagens na triagem: {len(triagem)}")
print(f"Imagens no dataset selecionado: {len(dataset)}")


In [ ]:
def normalizar(img):
    img = img.astype(np.float32)

    minimo = np.nanpercentile(img, 2)
    maximo = np.nanpercentile(img, 98)

    if maximo <= minimo:
        return np.zeros_like(img, dtype=np.float32)

    img = np.clip(img, minimo, maximo)

    return (img - minimo) / (maximo - minimo)


def normalizar_conjunta(canais):
    imagem = np.dstack(canais).astype(np.float32)

    minimo = np.nanpercentile(imagem, 2)
    maximo = np.nanpercentile(imagem, 98)

    if maximo <= minimo:
        return np.zeros_like(imagem, dtype=np.float32)

    imagem = np.clip(imagem, minimo, maximo)

    return (imagem - minimo) / (maximo - minimo)


def calcular_indice(banda_a, banda_b):
    banda_a = banda_a.astype(np.float32)
    banda_b = banda_b.astype(np.float32)

    denominador = banda_a + banda_b

    return np.divide(
        banda_a - banda_b,
        denominador,
        out=np.zeros_like(banda_a, dtype=np.float32),
        where=np.abs(denominador) > 1e-6
    )


def abrir_imagem(caminho):
    with rasterio.open(caminho) as src:
        bandas = {
            "B02": src.read(1),
            "B03": src.read(2),
            "B04": src.read(3),
            "B08": src.read(4),
            "B11": src.read(5),
            "B12": src.read(6)
        }

    return bandas


In [ ]:
def mostrar_imagem(bandas, titulo=""):
    rgb_natural = normalizar_conjunta([
        bandas["B04"],
        bandas["B03"],
        bandas["B02"]
    ])

    rgb_contrastado = np.dstack([
        normalizar(bandas["B04"]),
        normalizar(bandas["B03"]),
        normalizar(bandas["B02"])
    ])

    falsa_cor_nir = np.dstack([
        normalizar(bandas["B08"]),
        normalizar(bandas["B04"]),
        normalizar(bandas["B03"])
    ])

    composicao_swir = np.dstack([
        normalizar(bandas["B11"]),
        normalizar(bandas["B08"]),
        normalizar(bandas["B04"])
    ])

    ndwi = calcular_indice(
        bandas["B03"],
        bandas["B08"]
    )

    mndwi = calcular_indice(
        bandas["B03"],
        bandas["B11"]
    )

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(18, 12)
    )

    axes[0, 0].imshow(rgb_natural)
    axes[0, 0].set_title("RGB natural")

    axes[0, 1].imshow(rgb_contrastado)
    axes[0, 1].set_title("RGB contrastado")

    axes[0, 2].imshow(falsa_cor_nir)
    axes[0, 2].set_title("Falsa cor — NIR")

    axes[1, 0].imshow(composicao_swir)
    axes[1, 0].set_title("Composição SWIR")

    imagem_ndwi = axes[1, 1].imshow(
        ndwi,
        cmap="BrBG",
        vmin=-1,
        vmax=1
    )
    axes[1, 1].set_title("NDWI")
    fig.colorbar(
        imagem_ndwi,
        ax=axes[1, 1],
        fraction=0.046,
        pad=0.04
    )

    imagem_mndwi = axes[1, 2].imshow(
        mndwi,
        cmap="BrBG",
        vmin=-1,
        vmax=1
    )
    axes[1, 2].set_title("MNDWI")
    fig.colorbar(
        imagem_mndwi,
        ax=axes[1, 2],
        fraction=0.046,
        pad=0.04
    )

    for eixo in axes.ravel():
        eixo.axis("off")

    if titulo:
        fig.suptitle(
            titulo,
            fontsize=18,
            y=0.98
        )

    plt.tight_layout(
        rect=[0, 0, 1, 0.96]
    )

    plt.show()


In [ ]:
def corrigir_classificacao(
    id_imagem,
    possui_banco,
    qualidade="boa",
    usar_dataset=True,
    observacao="corrigido durante a revisão"
):
    global triagem

    id_imagem = str(id_imagem)

    filtro = triagem["id"].astype(str).eq(id_imagem)

    if not filtro.any():
        print(f"ID {id_imagem} não encontrado no triagem.csv.")
        return False

    triagem.loc[filtro, "possui_banco"] = possui_banco
    triagem.loc[filtro, "qualidade"] = qualidade
    triagem.loc[filtro, "usar_dataset"] = usar_dataset
    triagem.loc[filtro, "observacao"] = observacao

    triagem.to_csv(
        caminho_triagem,
        index=False
    )

    return True


In [ ]:
# True: revisa todas as imagens do dataset selecionado
# False: revisa apenas uma amostra aleatória
revisar_todas = True

if revisar_todas:
    imagens_revisao = dataset.copy()
else:
    imagens_revisao = dataset.sample(
        n=min(20, len(dataset)),
        random_state=42
    ).copy()

imagens_revisao = imagens_revisao.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

for indice, linha in imagens_revisao.iterrows():
    clear_output(wait=True)
    plt.close("all")

    id_imagem = str(linha["id"]).replace(".0", "")
    caminho = f"../data/raw/images/{id_imagem}.tif"

    if not os.path.exists(caminho):
        print(f"Arquivo não encontrado: {caminho}")
        input("Pressione ENTER para continuar...")
        continue

    registro = triagem[
        triagem["id"].astype(str).eq(id_imagem)
    ]

    if registro.empty:
        print(f"ID {id_imagem} não encontrado na triagem.")
        input("Pressione ENTER para continuar...")
        continue

    registro = registro.iloc[0]

    banco_atual = registro["possui_banco"]
    qualidade_atual = registro["qualidade"]
    uso_atual = registro["usar_dataset"]

    bandas = abrir_imagem(caminho)

    mostrar_imagem(
        bandas,
        titulo=(
            f"Imagem {id_imagem} | "
            f"{indice + 1} de {len(imagens_revisao)}\n"
            f"Classificação atual: banco={banco_atual} | "
            f"qualidade={qualidade_atual} | "
            f"usar={uso_atual}"
        )
    )

    print("\nEscolha uma opção:")
    print("1 - Manter a classificação atual")
    print("2 - Corrigir para: POSSUI banco")
    print("3 - Corrigir para: NÃO possui banco")
    print("4 - Marcar como imagem ruim e não utilizar")
    print("5 - Marcar para revisão posterior")
    print("q - Encerrar")

    while True:
        escolha = input("Escolha: ").strip().lower()

        if escolha in {"1", "2", "3", "4", "5", "q"}:
            break

        print("Opção inválida.")

    if escolha == "q":
        print("Revisão encerrada.")
        break

    if escolha == "1":
        pass

    elif escolha == "2":
        corrigir_classificacao(
            id_imagem=id_imagem,
            possui_banco=True,
            qualidade="boa",
            usar_dataset=True,
            observacao="corrigido para possui banco"
        )

    elif escolha == "3":
        corrigir_classificacao(
            id_imagem=id_imagem,
            possui_banco=False,
            qualidade="boa",
            usar_dataset=True,
            observacao="corrigido para não possui banco"
        )

    elif escolha == "4":
        corrigir_classificacao(
            id_imagem=id_imagem,
            possui_banco=False,
            qualidade="ruim",
            usar_dataset=False,
            observacao="imagem descartada durante a revisão"
        )

    elif escolha == "5":
        corrigir_classificacao(
            id_imagem=id_imagem,
            possui_banco=banco_atual,
            qualidade="revisar",
            usar_dataset=False,
            observacao="revisar classificação posteriormente"
        )

plt.close("all")
clear_output(wait=True)

print("Revisão finalizada.")


In [ ]:
# Recria o dataset selecionado após as correções

triagem = pd.read_csv(caminho_triagem)

triagem["possui_banco"] = (
    triagem["possui_banco"]
    .astype(str)
    .str.lower()
    .map({"true": True, "false": False})
)

triagem["usar_dataset"] = (
    triagem["usar_dataset"]
    .astype(str)
    .str.lower()
    .map({"true": True, "false": False})
)

positivas = triagem[
    (triagem["possui_banco"] == True)
    & (triagem["qualidade"] == "boa")
    & (triagem["usar_dataset"] == True)
].copy()

negativas = triagem[
    (triagem["possui_banco"] == False)
    & (triagem["qualidade"] == "boa")
    & (triagem["usar_dataset"] == True)
].copy()

quantidade_negativas = min(50, len(negativas))

negativas_selecionadas = negativas.sample(
    n=quantidade_negativas,
    random_state=42
)

dataset_selecionado = pd.concat(
    [positivas, negativas_selecionadas],
    ignore_index=True
)

dataset_selecionado = dataset_selecionado.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

dataset_selecionado.to_csv(
    caminho_dataset,
    index=False
)

print(f"Positivas: {len(positivas)}")
print(f"Negativas selecionadas: {len(negativas_selecionadas)}")
print(f"Total final: {len(dataset_selecionado)}")
print(f"Arquivo atualizado: {caminho_dataset}")
